# Step 2 — Sparkov Data Collection and Understanding

This notebook creates a reproducible, full-row profile of the raw Sparkov credit-card transaction files. It does not modify the source CSVs. The reusable implementation lives in `src/data_profile.py`; configuration lives in `configs/data_profile.json`.

**Test-data policy:** the test file is profiled only to document schema, integrity, class prevalence, and distribution shift. Its labels must not be used for feature, model, hyperparameter, or threshold selection.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'configs' / 'data_profile.json').exists():
    raise RuntimeError(f'Run from the project root or notebooks directory; got {Path.cwd()}')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_profile import build_profile

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_colwidth', 100)
PROJECT_ROOT

PosixPath('/Users/rcorpin002/Projects/personal/aim/Capstone')

## 1. Configuration and source-file check

Paths, profiling columns, target, timestamp, and chunk size are versioned rather than hidden in notebook state.

In [2]:
config_path = PROJECT_ROOT / 'configs' / 'data_profile.json'
config = json.loads(config_path.read_text(encoding='utf-8'))
source_files = {name: PROJECT_ROOT / relative for name, relative in config['files'].items()}

source_check = pd.DataFrame([
    {
        'split': name,
        'path': str(path.relative_to(PROJECT_ROOT)),
        'exists': path.exists(),
        'size_mb': round(path.stat().st_size / 1024**2, 2) if path.exists() else None,
    }
    for name, path in source_files.items()
])
display(source_check)
assert source_check['exists'].all(), 'One or more source files are missing.'

,split,path,exists,size_mb
0,train,data/raw/sparkov/fraudTrain.csv,True,334.97
1,test,data/raw/sparkov/fraudTest.csv,True,143.39


## 2. Raw schema preview

Only five rows are loaded here. Full-file counts and statistics are generated by the chunked profiler in the next section. Direct identifiers shown in the raw schema will not be used as raw predictive features.

In [3]:
raw_preview = pd.read_csv(source_files['train'], nrows=5)
display(raw_preview.drop(columns=['first', 'last', 'street', 'cc_num', 'trans_num']))
display(pd.DataFrame({'column': raw_preview.columns, 'inferred_dtype': raw_preview.dtypes.astype(str).values}))

,Unnamed: 0,trans_date_trans_time,merchant,category,amt,gender,city,state,zip,lat,long,city_pop,job,dob,unix_time,merch_lat,merch_long,is_fraud
0,0,2019-01-01 00:00:18,"fraud_Rippin, Kub and Mann",misc_net,4.97,F,Moravian Falls,NC,28654,36.0788,-81.1781,3495,"Psychologist, counselling",1988-03-09,1325376018,36.011293,-82.048315,0
1,1,2019-01-01 00:00:44,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,F,Orient,WA,99160,48.8878,-118.2105,149,Special educational needs teacher,1978-06-21,1325376044,49.159047,-118.186462,0
2,2,2019-01-01 00:00:51,fraud_Lind-Buckridge,entertainment,220.11,M,Malad City,ID,83252,42.1808,-112.2620,4154,Nature conservation officer,1962-01-19,1325376051,43.150704,-112.154481,0
3,3,2019-01-01 00:01:16,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,M,Boulder,MT,59632,46.2306,-112.1138,1939,Patent attorney,1967-01-12,1325376076,47.034331,-112.561071,0
4,4,2019-01-01 00:03:06,fraud_Keeling-Crist,misc_pos,41.96,M,Doe Hill,VA,24433,38.4207,-79.4629,99,Dance movement psychotherapist,1986-03-28,1325376186,38.674999,-78.632459,0


,column,inferred_dtype
0,Unnamed: 0,int64
1,trans_date_trans_time,str
2,cc_num,int64
3,merchant,str
4,category,str
5,amt,float64
6,first,str
7,last,str
8,gender,str
9,street,str


## 3. Generate the full reproducible profile

This cell reads every row in chunks, computes exact missing-value and class counts, checks transaction-ID uniqueness and cross-split overlap, calculates descriptive statistics, and applies Tukey's 1.5×IQR rule to amount and city population. IQR flags describe distribution tails; they are not automatic deletion rules.

In [4]:
profile = build_profile(PROJECT_ROOT)
step_2_dir = PROJECT_ROOT / 'deliverables' / '02_data_understanding'
profile_path = step_2_dir / 'data_profile.json'
dictionary_path = step_2_dir / 'data_dictionary.csv'
print(f'Wrote: {profile_path.relative_to(PROJECT_ROOT)}')
print(f'Wrote: {dictionary_path.relative_to(PROJECT_ROOT)}')

Wrote: deliverables/02_data_understanding/data_profile.json
Wrote: deliverables/02_data_understanding/data_dictionary.csv


## 4. Dataset overview and integrity

In [5]:
overview_rows = []
for split, values in profile['splits'].items():
    overview_rows.append({
        'split': split,
        'rows': values['rows'],
        'columns': values['columns_count'],
        'fraud_cases': values['fraud_cases'],
        'fraud_rate_pct': 100 * values['fraud_rate'],
        'start': values['timestamp_min'],
        'end': values['timestamp_max'],
        'missing_values': values['total_missing_values'],
        'duplicate_transaction_ids': values['duplicate_transaction_ids'],
        'sequential_export_index': values['raw_index_is_sequential'],
    })
overview = pd.DataFrame(overview_rows)
display(overview.style.format({'rows': '{:,.0f}', 'fraud_cases': '{:,.0f}', 'fraud_rate_pct': '{:.4f}%'}))
print('Cross-split transaction-ID overlap:', profile['combined']['cross_split_transaction_id_overlap'])

,split,rows,columns,fraud_cases,fraud_rate_pct,start,end,missing_values,duplicate_transaction_ids,sequential_export_index
0,train,"1,296,675",23,"7,506",0.5789%,2019-01-01T00:00:18,2020-06-21T12:13:37,0,0,True
1,test,"555,719",23,"2,145",0.3860%,2020-06-21T12:14:25,2020-12-31T23:59:34,0,0,True


Cross-split transaction-ID overlap: 0


## 5. Feature types, cardinality, and allowed values

In [6]:
train = profile['splits']['train']
feature_summary = []
for column in train['columns']:
    categorical = train['categorical_summary'].get(column)
    feature_summary.append({
        'column': column,
        'raw_dtype': train['raw_dtypes'][column],
        'missing_train': train['missing_counts'][column],
        'cardinality_if_profiled': categorical['cardinality'] if categorical else None,
        'allowed_values_if_small': ', '.join(categorical['allowed_values']) if categorical and categorical['allowed_values'] else None,
    })
display(pd.DataFrame(feature_summary))

,column,raw_dtype,missing_train,cardinality_if_profiled,allowed_values_if_small
0,Unnamed: 0,int64,0,NaN,NaN
1,trans_date_trans_time,str,0,NaN,NaN
2,cc_num,int64,0,983.0,NaN
3,merchant,str,0,693.0,NaN
4,category,str,0,14.0,"entertainment, food_dining, gas_transport, grocery_net, grocery_pos, health_fitness, home, kids_..."
5,amt,float64,0,NaN,NaN
6,first,str,0,352.0,NaN
7,last,str,0,481.0,NaN
8,gender,str,0,2.0,"F, M"
9,street,str,0,983.0,NaN


## 6. Numeric distributions and IQR flags

Extreme transaction amounts may be genuine and predictive. They will be investigated and transformed or robustly modelled where appropriate, not removed merely for exceeding an IQR fence. Geographic coordinates are validated against physical bounds rather than treated as univariate outliers.

In [7]:
for split, values in profile['splits'].items():
    print(f'--- {split.upper()} NUMERIC SUMMARY ---')
    numeric = pd.DataFrame(values['numeric_summary']['descriptive_statistics']).T
    display(numeric)
    outliers = pd.DataFrame(values['numeric_summary']['iqr_outliers']).T
    display(outliers[['lower_fence', 'upper_fence', 'count', 'rate', 'interpretation']])

--- TRAIN NUMERIC SUMMARY ---


,count,mean,std,min,1%,25%,median,75%,99%,max
amt,1296675.0,7.035104e+01,1.603160e+02,1.000000e+00,1.260000e+00,9.650000e+00,4.752000e+01,8.314000e+01,5.459926e+02,2.894890e+04
city_pop,1296675.0,8.882444e+04,3.019564e+05,2.300000e+01,5.300000e+01,7.430000e+02,2.456000e+03,2.032800e+04,1.577385e+06,2.906700e+06
lat,1296675.0,3.853762e+01,5.075808e+00,2.002710e+01,2.647220e+01,3.462050e+01,3.935430e+01,4.194040e+01,4.847860e+01,6.669330e+01
long,1296675.0,-9.022634e+01,1.375908e+01,-1.656723e+02,-1.230614e+02,-9.679800e+01,-8.747690e+01,-8.015800e+01,-7.034570e+01,-6.795030e+01
unix_time,1296675.0,1.349244e+09,1.284128e+07,1.325376e+09,1.326022e+09,1.338751e+09,1.349250e+09,1.359385e+09,1.371363e+09,1.371817e+09
merch_lat,1296675.0,3.853734e+01,5.109788e+00,1.902779e+01,2.638941e+01,3.473357e+01,3.936568e+01,4.195716e+01,4.857791e+01,6.751027e+01
merch_long,1296675.0,-9.022646e+01,1.377109e+01,-1.666712e+02,-1.235620e+02,-9.689728e+01,-8.743839e+01,-8.023680e+01,-7.038504e+01,-6.695090e+01


,lower_fence,upper_fence,count,rate,interpretation
amt,-100.585,193.375,67290,0.051894,"distributional flag, not automatic evidence of invalid data"
city_pop,-28634.5,49705.5,242674,0.187151,"distributional flag, not automatic evidence of invalid data"


--- TEST NUMERIC SUMMARY ---


,count,mean,std,min,1%,25%,median,75%,99%,max
amt,555719.0,6.939281e+01,1.567459e+02,1.000000e+00,1.260000e+00,9.630000e+00,4.729000e+01,8.301000e+01,5.198546e+02,2.276811e+04
city_pop,555719.0,8.822189e+04,3.003909e+05,2.300000e+01,5.300000e+01,7.410000e+02,2.408000e+03,1.968500e+04,1.577385e+06,2.906700e+06
lat,555719.0,3.854325e+01,5.061336e+00,2.002710e+01,2.647220e+01,3.466890e+01,3.937160e+01,4.189480e+01,4.847860e+01,6.568990e+01
long,555719.0,-9.023133e+01,1.372178e+01,-1.656723e+02,-1.230614e+02,-9.679800e+01,-8.747690e+01,-8.017520e+01,-7.034570e+01,-6.795030e+01
unix_time,555719.0,1.380679e+09,5.201104e+06,1.371817e+09,1.371936e+09,1.376029e+09,1.380762e+09,1.385867e+09,1.388391e+09,1.388534e+09
merch_lat,555719.0,3.854280e+01,5.095829e+00,1.902742e+01,2.640630e+01,3.475530e+01,3.937659e+01,4.195416e+01,4.857647e+01,6.667930e+01
merch_long,555719.0,-9.023138e+01,1.373307e+01,-1.666716e+02,-1.235522e+02,-9.690513e+01,-8.744520e+01,-8.026464e+01,-7.042688e+01,-6.695203e+01


,lower_fence,upper_fence,count,rate,interpretation
amt,-100.44,193.08,27778,0.049986,"distributional flag, not automatic evidence of invalid data"
city_pop,-27675.0,48101.0,104151,0.187417,"distributional flag, not automatic evidence of invalid data"


## 7. Complete data dictionary

In [8]:
data_dictionary = pd.read_csv(dictionary_path)
display(data_dictionary)

,variable,role,type,unit_or_format,allowed_values_or_range,description
0,Unnamed: 0,raw row index,integer identifier,sequential integer,0 to rows_in_file - 1,CSV export index; not a predictive feature and will be removed.
1,trans_date_trans_time,predictor / ordering,datetime,YYYY-MM-DD HH:MM:SS,observed dataset period,Transaction timestamp used for chronological splitting and temporal features.
2,cc_num,entity identifier,integer identifier,card number,synthetic card identifiers,Synthetic cardholder account identifier; excluded as a raw model feature but usable to construct...
3,merchant,predictor,categorical string,merchant name,observed merchant categories,Synthetic merchant associated with the transaction.
4,category,predictor,categorical string,merchant category,observed category labels,Type of goods or services associated with the merchant.
5,amt,predictor / value KPI,continuous numeric,US dollars,non-negative transaction amount,Monetary value of the transaction.
6,first,direct identifier,categorical string,given name,synthetic names,Synthetic cardholder first name; excluded from modelling.
7,last,direct identifier,categorical string,family name,synthetic names,Synthetic cardholder last name; excluded from modelling.
8,gender,protected audit attribute,binary categorical,category,"F, M",Synthetic cardholder gender; retained for fairness auditing only and excluded from model inputs.
9,street,direct identifier,categorical string,street address,synthetic addresses,Synthetic cardholder street address; excluded from modelling.


## 8. Initial data-understanding conclusions

Conclusions in the executed notebook are supported by the generated profile rather than a sample. The written Step 2 report summarizes the findings and distinguishes data-quality defects from legitimate rare behavior.

Step 3 below performs development-only applied EDA, creates strict-past temporal and behavioral features, formalizes preprocessing, and evaluates feature selection, diagnostic importance, and PCA.

# Step 3 — Applied EDA, Preprocessing, and Feature Engineering

The supplied test file remains untouched. The final 20% of the original training period is reserved as chronological validation; the reusable Step 3 analysis fits transformations and produces findings from development data only.

In [9]:
from src.step3_analysis import run as run_step3

step3_summary = run_step3(PROJECT_ROOT)

{
  "random_seed": 42,
  "raw_training_rows": 1296675,
  "development_fraction_requested": 0.8,
  "split_boundary_timestamp": "2020-03-06T07:16:43",
  "development": {
    "rows": 1037340,
    "start": "2019-01-01T00:00:18",
    "end": "2020-03-06T07:15:17",
    "fraud_cases": 5968,
    "fraud_rate": 0.005753176393467908
  },
  "validation": {
    "rows": 259335,
    "start": "2020-03-06T07:16:43",
    "end": "2020-06-21T12:13:37",
    "fraud_cases": 1538,
    "fraud_rate": 0.005930553145545337,
    "policy": "held out from Step 3 relationship EDA, selection, importance, and PCA"
  },
  "cleaning": {
    "missing_values_removed": 0,
    "duplicate_transaction_ids_removed": 0,
    "outlier_rows_removed": 0,
    "outlier_strategy": "retain; log-transform amount and city population; robust tree models evaluated later"
  },
  "diagnostic_sample": {
    "rows": 125968,
    "fraud_cases": 5968,
    "purpose": "feature ranking and diagnostic importance only; not unbiased performance estimatio

## Step 3 results and saved evidence

All figures and tables are saved under `deliverables/03_preprocessing_eda_features/`. The enriched sample below is used only for exploratory feature ranking and diagnostic importance, never for unbiased performance estimation.

In [10]:
step3_dir = PROJECT_ROOT / 'deliverables' / '03_preprocessing_eda_features'
selected_features = pd.read_csv(step3_dir / 'tables' / 'selected_features_filter_method.csv')
model_importance = pd.read_csv(step3_dir / 'tables' / 'extra_trees_feature_importance.csv')
pca_variance = pd.read_csv(step3_dir / 'tables' / 'pca_explained_variance.csv')
display(selected_features.head(25))
display(model_importance.head(20))
display(pca_variance)

,rank,feature,mutual_information
0,1,numeric__log_amt,0.113707
1,2,numeric__log_amount_to_prior_mean_ratio,0.076517
2,3,one_hot__amount_band_over_500,0.060162
3,4,numeric__hour_cos,0.037952
4,5,numeric__long,0.037330
5,6,numeric__lat,0.036560
6,7,numeric__log_city_pop,0.032476
7,8,frequency__zip_frequency,0.026769
8,9,frequency__city_frequency,0.024021
9,10,numeric__hour_sin,0.021030


,feature,importance
0,numeric__log_amt,0.148952
1,one_hot__amount_band_over_500,0.131070
2,numeric__hour_cos,0.086685
3,one_hot__amount_band_50_100,0.085368
4,numeric__log_amount_to_prior_mean_ratio,0.078805
5,numeric__hour_sin,0.038210
6,one_hot__hour_band_afternoon,0.031952
7,one_hot__amount_band_under_10,0.030806
8,one_hot__amount_band_100_500,0.028964
9,one_hot__category_grocery_pos,0.028583


,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.142708,0.142708
1,2,0.109383,0.252090
2,3,0.103041,0.355132
3,4,0.099163,0.454295
4,5,0.091824,0.546119
5,6,0.078249,0.624368
6,7,0.075803,0.700171
7,8,0.053583,0.753754
8,9,0.049654,0.803408
9,10,0.049032,0.852440


## Step 3 decisions

- Remove no valid row solely because it is an IQR outlier; use transformations and model comparison.
- Exclude direct identifiers and keep gender/age for audit only in the primary branch.
- Use strict-past card, merchant, and card-category behavior; simultaneous transactions cannot leak into one another.
- Carry both the full engineered set and a top-25 mutual-information branch into Step 4.
- Retain PCA as a comparison branch only because 13 of 20 components are required for 95% variance and PC1–PC2 do not separate fraud cleanly.
- Apply SHAP/PDP to the models selected through chronological validation, not to the diagnostic model as if it were final.